### Basic STM

In [ ]:
class SimpleSTM:
    def __init__(self):
        # Stores conversation history as (user, bot) pairs
        self.memory = []

    def add_interaction(self, user_input, llm_response):
        """Stores user question and LLM response in memory."""
        self.memory.append((user_input, llm_response))

    def get_context(self):
        """Formats stored memory as conversation context for LLM."""
        return "\n".join([f"User: {q}\nLLM: {a}" for q, a in self.memory])


# Example Usage
stm = SimpleSTM()
stm.add_interaction("What is AI?", "AI stands for Artificial Intelligence.")
stm.add_interaction("How does machine learning work?", "Machine learning is a subset of AI.")
context = stm.get_context()
print(context)

### Advanced STM

In [ ]:
tools = [{
    "type": "function",
    "function": {
        "name": "send_email",
        "description": "Send an email to a given recipient with a subject and message.",
        "parameters": {
            "type": "object",
            "properties": {
                "to": {
                    "type": "string",
                    "description": "The recipient email address."
                },
                "subject": {
                    "type": "string",
                    "description": "Email subject line."
                },
                "body": {
                    "type": "string",
                    "description": "Body of the email message."
                }
            },
            "required": [
                "to",
                "subject",
                "body"
            ],
            "additionalProperties": False
        },
        "strict": True
    }
}]

In [ ]:
from dataclasses import dataclass, field
from typing import List, Tuple, Callable, Optional, Dict
import re

# ---------- Pluggable hooks (replace with your production versions) ----------

def default_token_count(text: str) -> int:
    """
    Rough token estimator: ~1.3× words for plain text; a bit higher for code/JSON.
    Good enough to budget; swap for model tokenizer (e.g., tiktoken) in prod.
    """
    words = len(re.findall(r"\S+", text))
    # Heuristic bump for heavy punctuation / JSON-ish content
    punct_ratio = len(re.findall(r"[{}[\],:;]", text)) / max(1, len(text))
    coef = 1.3 + (0.2 if punct_ratio > 0.02 else 0.0)
    return int(coef * words)

def default_summarize(text: str, target_tokens: int) -> str:
    """
    Minimal abstractive stub: keeps the most informative sentences by length/density,
    then trims at a sentence boundary near target_tokens. Replace with your LLM call.
    """
    # Split into sentences and rank by 'informativeness' ≈ alphabetic characters count
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    ranked = sorted(sents, key=lambda s: len(re.findall(r"[A-Za-z0-9]", s)), reverse=True)
    out = []
    for s in ranked:
        out.append(s)
        if default_token_count(" ".join(out)) >= target_tokens:
            break
    summary = " ".join(out)
    # Soft trim on sentence boundary if we overshot
    while default_token_count(summary) > target_tokens and len(out) > 1:
        out.pop()
        summary = " ".join(out)
    return summary

def default_topic_labels(text: str) -> List[str]:
    """
    Tiny topic stub; replace with embedding/classifier. Multi-label allowed.
    """
    t = text.lower()
    labels = []
    for kw, lab in [
        (["flight","hotel","travel","itinerary","visa"], "travel"),
        (["invoice","budget","tax","payment","pricing","finance"], "finance"),
        (["bug","deploy","api","stacktrace","error","code"], "engineering"),
        (["policy","contract","law","compliance"], "legal"),
        (["order","return","support","helpdesk","ticket"], "support"),
    ]:
        if any(k in t for k in kw): labels.append(lab)
    return labels or ["general"]

def default_topic_drift(prev_topics: List[str], new_topics: List[str], overlap_min: int = 1) -> bool:
    """
    Returns True if we consider the new turn to have drifted far enough to start a new session.
    """
    return len(set(prev_topics) & set(new_topics)) < overlap_min


# ----------------------------- Memory structures -----------------------------

@dataclass
class Turn:
    user: str
    agent: str
    topics: List[str]

    def as_text(self) -> str:
        return f"User: {self.user}\nAgent: {self.agent}"

@dataclass
class Session:
    id: int
    turns: List[Turn] = field(default_factory=list)
    rolling_summary: str = ""  # compressed history older than the recent buffer
    summary_topics: List[str] = field(default_factory=list)

@dataclass
class ShortTermMemory:
    # Policy knobs (tune per app/model)
    recent_turns: int = 8                     # preserve last N turns verbatim
    summary_target_tokens: int = 300          # target size of rolling summary
    max_context_tokens: int = 2000            # overall prompt budget for memory
    cut_guard_tokens: int = 200               # spare room to avoid boundary overflow

    # Pluggable components
    tokenize: Callable[[str], int] = default_token_count
    summarize: Callable[[str, int], str] = default_summarize
    topic_labels: Callable[[str], List[str]] = default_topic_labels
    topic_drift: Callable[[List[str], List[str]], bool] = default_topic_drift

    # Internal state
    sessions: List[Session] = field(default_factory=lambda: [Session(id=1)])
    _next_session_id: int = 2

    # ---------------------- Public API ----------------------

    def add_turn(self, user_text: str, agent_text: str):
        """Append a turn to the active session, updating topics and rolling summary as needed."""
        sess = self.sessions[-1]
        topics = list(set(self.topic_labels(user_text) + self.topic_labels(agent_text)))
        sess.turns.append(Turn(user=user_text, agent=agent_text, topics=topics))

        # Maintain rolling summary when turns exceed recent buffer
        if len(sess.turns) > self.recent_turns:
            old_block = "\n\n".join(t.as_text() for t in sess.turns[:-self.recent_turns])
            sess.rolling_summary = self._summarize_and_merge(sess.rolling_summary, old_block)
            # Keep only the recent buffer as raw turns
            sess.turns = sess.turns[-self.recent_turns:]
            # Track topics represented in the summary
            sess.summary_topics = sorted(list(set(
                (sess.summary_topics or []) + [lab for tr in sess.turns for lab in tr.topics]
            )))

    def maybe_start_new_session(self, new_user_text: str):
        """
        Detect topic drift vs. current session tail; if drift, start a new session.
        """
        new_topics = self.topic_labels(new_user_text)
        prev_topics = self._current_topics()
        if prev_topics and self.topic_drift(prev_topics, new_topics):
            self.sessions.append(Session(id=self._next_session_id))
            self._next_session_id += 1

    def build_context(self, user_text: str) -> str:
        """
        Compose a memory context under the token budget with:
        1) topic filtering (TAM),
        2) recent buffer verbatim,
        3) rolling summary,
        4) token-aware final trim on sentence boundary.
        """
        sess = self.sessions[-1]
        query_topics = self.topic_labels(user_text)

        recent = [t for t in sess.turns if self._overlaps(t.topics, query_topics)]
        # If nothing matches, fall back to last few turns to keep fluency
        if not recent:
            recent = sess.turns[-min(self.recent_turns, len(sess.turns)):]

        recent_text = "\n\n".join(t.as_text() for t in recent)
        parts = []
        budget = self.max_context_tokens - self.cut_guard_tokens

        # Prefer recent verbatim
        if recent_text:
            parts.append(recent_text)

        # Add rolling summary if topical and budget allows
        if sess.rolling_summary and self._overlaps(sess.summary_topics or query_topics, query_topics):
            if self.tokenize("\n\n".join(parts)) + self.tokenize(sess.rolling_summary) <= budget:
                parts.append(sess.rolling_summary)
            else:
                # Fit a re-summarized version if necessary
                remaining = max(0, budget - self.tokenize("\n\n".join(parts)))
                if remaining > 60:  # only if we have room for a meaningful summary
                    parts.append(self.summarize(sess.rolling_summary, remaining))

        context = "\n\n".join(parts).strip()

        # Final protective trim on sentence/paragraph boundary
        context = self._soft_trim(context, budget)
        return context

    # ---------------------- Helpers ----------------------

    def _summarize_and_merge(self, existing_summary: str, new_block_raw: str) -> str:
        if not new_block_raw.strip():
            return existing_summary
        merged = (existing_summary + "\n\n" + new_block_raw).strip() if existing_summary else new_block_raw
        return self.summarize(merged, self.summary_target_tokens)

    def _soft_trim(self, text: str, budget: int) -> str:
        if self.tokenize(text) <= budget:
            return text
        # Trim at paragraph or sentence boundary
        paras = re.split(r'\n\s*\n', text)
        out = []
        for p in paras:
            candidate = ("\n\n".join(out + [p])).strip()
            if self.tokenize(candidate) > budget:
                break
            out.append(p)
        if out:
            return "\n\n".join(out).strip()
        # Fallback: cut at nearest sentence boundary
        sents = re.split(r'(?<=[.!?])\s+', text)
        out = []
        for s in sents:
            candidate = (" ".join(out + [s])).strip()
            if self.tokenize(candidate) > budget:
                break
            out.append(s)
        return " ".join(out).strip()

    def _current_topics(self) -> List[str]:
        sess = self.sessions[-1]
        tail = sess.turns[-min(3, len(sess.turns)):]  # look at a short tail
        topics = set()
        for t in tail:
            topics.update(t.topics)
        return sorted(list(topics))

    @staticmethod
    def _overlaps(a: List[str], b: List[str]) -> bool:
        return bool(set(a) & set(b))


# ----------------------------- Example usage -----------------------------
if __name__ == "__main__":
    stm = ShortTermMemory(
        recent_turns=6,
        summary_target_tokens=250,
        max_context_tokens=1500
        # Plug your tokenizer/summarizer/classifier here if desired
    )

    # Conversation 1 (engineering topic)
    stm.maybe_start_new_session("I'm seeing a 500 error after deploy. Can you help?")
    stm.add_turn("I'm seeing a 500 error after deploy. Can you help?",
                 "Let's check the API logs and recent config changes.")
    stm.add_turn("Logs show a null pointer in ProductController.",
                 "Roll back the last commit; also add a null check and redeploy.")

    # Conversation 2 (topic drift → new session)
    stm.maybe_start_new_session("Plan a 5-day trip to Italy on a $2,000 budget.")
    stm.add_turn("Plan a 5-day trip to Italy on a $2,000 budget.",
                 "Start with Rome & Florence; off-peak trains keep costs low.")
    stm.add_turn("Add a day trip to Siena; any food tips?",
                 "Trattoria style places near Piazza del Campo; try pici pasta.")

    # Build context for a new user query (travel)
    query = "Book mid-range hotels near Termini and Santa Maria Novella."
    context = stm.build_context(query)
    print("----- MEMORY CONTEXT (to send with the LLM prompt) -----\n")
    print(context)
